Normaliza nombres de columnas: minúsculas, sin espacios ni acentos.

In [9]:
import os
import pandas as pd

ruta_excel = "../data/raw/Sales report completa.xlsx"
df = pd.read_excel(ruta_excel)
print(df.head(0))


Empty DataFrame
Columns: [Número Venta, Mes Salida, Dia Salida, Año Salida, Mes Entrega, Dia Entrega, Año Entrega, Método Envio, Número Cliente, Nombre Cliente, Segmento, Ciudad, Estado, País, ID Producto, Ventas, Cantidad, Descuento, Utilidad, Costo Envío, Prioridad Envio]
Index: []

[0 rows x 21 columns]


In [10]:


def estandarizar_columnas(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
    )
    return df 
# Aplicar la función al dataframe que ya cargaste
df = estandarizar_columnas(df)

# Mostrar las nuevas columnas estandarizadas
print(df.columns)

Index(['numero_venta', 'mes_salida', 'dia_salida', 'ano_salida', 'mes_entrega',
       'dia_entrega', 'ano_entrega', 'metodo_envio', 'numero_cliente',
       'nombre_cliente', 'segmento', 'ciudad', 'estado', 'pais', 'id_producto',
       'ventas', 'cantidad', 'descuento', 'utilidad', 'costo_envio',
       'prioridad_envio'],
      dtype='object')


Elimina los espacios en blanco innecesarios de la columna "Método de envío" Ejemplo: " Standard  	Class " por "Standard Class"

In [11]:
def limpiar_espacios_metodo_envio(df: pd.DataFrame) -> pd.DataFrame:
    if 'metodo_envio' in df.columns:
        df['metodo_envio'] = df['metodo_envio'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
    else:
        print("La columna 'metodo_envio' no existe en el DataFrame.")
    return df

# Aplicar la función
df = limpiar_espacios_metodo_envio(df)

# Ver los valores únicos de la columna después de limpiar
print(df['metodo_envio'].unique())


['Standard Class' 'Second Class' 'First Class' 'Same Day']


Unificar Mes/Dia/Año de las fecha_envio y fecha_entrega

In [12]:
def convertir_fechas(df: pd.DataFrame) -> pd.DataFrame:
    # Convertir las columnas de año, mes y día a números enteros (permitiendo nulos)
    for col in ['ano_salida', 'mes_salida', 'dia_salida', 'ano_entrega', 'mes_entrega', 'dia_entrega']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')  # 'Int64' permite NaN

    # Crear columna de fecha_salida si existen las tres columnas necesarias
    if {'ano_salida', 'mes_salida', 'dia_salida'}.issubset(df.columns):
        df['fecha_salida'] = pd.to_datetime(
            df[['ano_salida', 'mes_salida', 'dia_salida']].rename(
                columns={
                    'ano_salida': 'year',
                    'mes_salida': 'month',
                    'dia_salida': 'day'
                }
            ),
            errors='coerce'
        )
        
    # Crear columna de fecha_entrega si existen las tres columnas necesarias
    if {'ano_entrega', 'mes_entrega', 'dia_entrega'}.issubset(df.columns):
        df['fecha_entrega'] = pd.to_datetime(
            df[['ano_entrega', 'mes_entrega', 'dia_entrega']].rename(
                columns={
                    'ano_entrega': 'year',
                    'mes_entrega': 'month',
                    'dia_entrega': 'day'
                }
            ),
            errors='coerce'
        )

    return df

# Aplicar la función al DataFrame
df = convertir_fechas(df)

# Verificar las nuevas columnas de fecha
print(df[['fecha_salida', 'fecha_entrega']].head())
print(df.columns)


  fecha_salida fecha_entrega
0   2011-01-01    2011-01-06
1   2011-01-01    2011-01-08
2   2011-01-01    2011-01-05
3   2011-01-01    2011-01-05
4   2011-01-01    2011-01-08
Index(['numero_venta', 'mes_salida', 'dia_salida', 'ano_salida', 'mes_entrega',
       'dia_entrega', 'ano_entrega', 'metodo_envio', 'numero_cliente',
       'nombre_cliente', 'segmento', 'ciudad', 'estado', 'pais', 'id_producto',
       'ventas', 'cantidad', 'descuento', 'utilidad', 'costo_envio',
       'prioridad_envio', 'fecha_salida', 'fecha_entrega'],
      dtype='object')


Agrega una columna con los dias_transcurridos" entre la Fecha Salida y Fecha 
Entrega. 

In [16]:
def calcular_duracion_envio(df: pd.DataFrame) -> pd.DataFrame:
    if {'fecha_salida', 'fecha_entrega'}.issubset(df.columns):
        df['duracion_envio'] = (df['fecha_entrega'] - df['fecha_salida']).dt.days
    else:
        print("Las columnas 'fecha_salida' y/o 'fecha_entrega' no existen en el DataFrame.")
    return df

# Aplicar la función al DataFrame
df = calcular_duracion_envio(df)

# Verificar resultado
print(df[['fecha_salida', 'fecha_entrega', 'duracion_envio']].head())


  fecha_salida fecha_entrega  duracion_envio
0   2011-01-01    2011-01-06               5
1   2011-01-01    2011-01-08               7
2   2011-01-01    2011-01-05               4
3   2011-01-01    2011-01-05               4
4   2011-01-01    2011-01-08               7


Agrega una columna con  id_venta unificando y pasando a mayusc: 2 iniciales de 
"País", "Año Salida" y "Número Venta" (ejemplo: Al-2011-2040). 


In [17]:
def crear_id_venta(df: pd.DataFrame) -> pd.DataFrame:
    if {'pais', 'ano_salida', 'numero_venta'}.issubset(df.columns):
        # Asegurar que los datos sean cadenas y manejar valores faltantes
        df['id_venta'] = (
            df['pais'].astype(str).str.strip().str[:2]      # Primeras 2 letras del país
            + '-' +
            df['ano_salida'].astype(str).str.strip()        # Año de salida
            + '-' +
            df['numero_venta'].astype(str).str.strip()      # Número de venta
        ).str.upper()  # Convertir todo a mayúsculas
    else:
        print("Faltan una o más columnas necesarias: 'pais', 'ano_salida', 'numero_venta'")
    return df

# Aplicar la función
df = crear_id_venta(df)

# Verificar las primeras filas
print(df[['pais', 'ano_salida', 'numero_venta', 'id_venta']].head())


        pais  ano_salida  numero_venta         id_venta
0    Algeria        2011          2040     AL-2011-2040
1  Australia        2011         47883    AU-2011-47883
2    Hungary        2011          1220     HU-2011-1220
3     Sweden        2011       3647632  SW-2011-3647632
4  Australia        2011         47883    AU-2011-47883
